# Real-Time Streaming Inference

## 1. Setup & Imports

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.ml import PipelineModel
from pyspark.ml.tuning import CrossValidatorModel
from pyspark.ml.feature import StringIndexerModel
import pickle
import os
import shutil
import glob
import time

In [3]:
try:
    spark.stop()
except:
    pass

spark = (
    SparkSession.builder
    .appName("high-performance-local-spark")
    .master("local[24]")
    .config("spark.driver.memory", "180g")
    .config("spark.driver.maxResultSize", "20g")
    .config("spark.sql.shuffle.partitions", "96")
    .config("spark.default.parallelism", "96")
    .config("spark.memory.fraction", "0.75")
    .config("spark.memory.storageFraction", "0.3")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark {spark.version} ready")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/03 02:30:17 WARN Utils: Your hostname, ml-ai-ubuntu-gpu-h200x1-141gb-atl1, resolves to a loopback address: 127.0.1.1; using 10.50.0.5 instead (on interface eth0)
26/05/03 02:30:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 02:30:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/03 02:30:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark 4.1.1 ready


## 2. Load Saved Models

In [4]:
# ── Feature engineering pipeline (3 stages) ──────────────────────────────────
# Stage 0: StringIndexer  → label_index
# Stage 1: VectorAssembler → features_raw_sel  (40 selected cols)
# Stage 2: StandardScaler  → features_selected
feature_pipeline = PipelineModel.load("models/feature_pipeline")

# ── Classifiers ──────────────────────────────────────────────────────────────
# XGBoost: F1=0.987 (best) — use rf_best if XGBoost causes issues in streaming
xgb_best = CrossValidatorModel.load("models/xgb_cv_model").bestModel
rf_best  = CrossValidatorModel.load("models/rf_cv_model").bestModel

# ── Feature metadata ─────────────────────────────────────────────────────────
with open("models/selected_features.pkl", "rb") as f:
    selected_features = pickle.load(f)

# ── Label map: index → class name ────────────────────────────────────────────
indexer_model = StringIndexerModel.load("models/string_indexer")
label_map = {float(i): name for i, name in enumerate(indexer_model.labels)}

print("Pipeline stages :", [type(s).__name__ for s in feature_pipeline.stages])
print("Selected features:", len(selected_features))
print("Label map        :", label_map)

Pipeline stages : ['StringIndexerModel', 'VectorAssembler', 'StandardScalerModel']
Selected features: 41
Label map        : {0.0: 'BruteForce', 1.0: 'WebAttack', 2.0: 'Botnet', 3.0: 'DoS/DDoS', 4.0: 'Normal', 5.0: 'Recon', 6.0: 'Infiltration', 7.0: 'Heartbleed'}


## 3. Schema & Streaming Source

In [5]:
# Read schema from existing clean data — incoming stream must match this
raw_schema = spark.read.parquet("data/clean/").schema
print(f"Input schema: {len(raw_schema.fields)} columns")
for field in raw_schema.fields:
    print(f"  {field.name}: {field.dataType}")

Input schema: 60 columns
  Destination Port: IntegerType()
  Flow Duration: IntegerType()
  Total Fwd Packets: IntegerType()
  Total Backward Packets: IntegerType()
  Total Length of Fwd Packets: IntegerType()
  Total Length of Bwd Packets: IntegerType()
  Fwd Packet Length Max: IntegerType()
  Fwd Packet Length Min: IntegerType()
  Fwd Packet Length Mean: DoubleType()
  Fwd Packet Length Std: DoubleType()
  Bwd Packet Length Max: IntegerType()
  Bwd Packet Length Min: IntegerType()
  Bwd Packet Length Mean: DoubleType()
  Bwd Packet Length Std: DoubleType()
  Flow Bytes/s: DoubleType()
  Flow Packets/s: DoubleType()
  Flow IAT Mean: DoubleType()
  Flow IAT Std: DoubleType()
  Flow IAT Max: IntegerType()
  Flow IAT Min: IntegerType()
  Fwd IAT Total: IntegerType()
  Fwd IAT Mean: DoubleType()
  Fwd IAT Std: DoubleType()
  Fwd IAT Max: IntegerType()
  Fwd IAT Min: IntegerType()
  Bwd IAT Total: IntegerType()
  Bwd IAT Mean: DoubleType()
  Bwd IAT Std: DoubleType()
  Bwd IAT Max: Integer

In [6]:
# Create directories
for d in ["data/stream_input", "data/stream_output", "data/stream_checkpoint"]:
    os.makedirs(d, exist_ok=True)

# Streaming source — picks up new parquet files dropped into stream_input/
stream_df = (
    spark.readStream
    .schema(raw_schema)
    .option("maxFilesPerTrigger", 1)   # 1 file per micro-batch (simulates one event window)
    .parquet("data/stream_input/")
)

print("Stream source ready.")
print("Drop parquet files into data/stream_input/ to trigger inference.")

Stream source ready.
Drop parquet files into data/stream_input/ to trigger inference.


## 4. Inference Logic (foreachBatch)

In [7]:
# Broadcast label map so it is available inside foreachBatch workers
label_map_bc = spark.sparkContext.broadcast(label_map)

@F.udf(StringType())
def decode_label(idx):
    """Convert numeric prediction index to class name."""
    return label_map_bc.value.get(float(idx), "Unknown")

In [8]:
def process_batch(batch_df, epoch_id):
    """
    Called for every micro-batch.
    Applies feature_pipeline → XGBoost → writes results.
    """
    if batch_df.rdd.isEmpty():
        print(f"[Batch {epoch_id}] Empty — skipping.")
        return

    n = batch_df.count()
    print(f"\n{'='*60}")
    print(f"[Batch {epoch_id}] {n:,} records received")
    t0 = time.time()

    # ── Step 1: Feature engineering ──────────────────────────────────────────
    # Applies StringIndexer → VectorAssembler (40 cols) → StandardScaler
    featured = feature_pipeline.transform(batch_df)

    # ── Step 2: Inference ─────────────────────────────────────────────────────
    # Switch xgb_best → rf_best if XGBoost raises issues in your environment
    predicted = xgb_best.transform(featured)

    # ── Step 3: Decode label index → class name ───────────────────────────────
    result = predicted.withColumn(
        "predicted_class", decode_label(F.col("prediction"))
    ).select(
        "Label",
        "label_index",
        "prediction",
        "predicted_class"
    )

    # ── Step 4: Console output ────────────────────────────────────────────────
    result.show(10, truncate=False)

    # ── Step 5: Accuracy (only meaningful when Label is present) ──────────────
    correct = result.filter(F.col("label_index") == F.col("prediction")).count()
    elapsed = time.time() - t0
    print(f"  Accuracy : {correct/n:.4f}  ({correct:,}/{n:,} correct)")
    print(f"  Latency  : {elapsed:.2f}s for {n:,} records  "
          f"({n/elapsed:,.0f} rec/s)")

    # ── Step 6: Per-class breakdown ───────────────────────────────────────────
    print("  Prediction distribution:")
    result.groupBy("predicted_class") \
          .count() \
          .orderBy("count", ascending=False) \
          .show(truncate=False)

    # ── Step 7: Persist results ───────────────────────────────────────────────
    result.write.mode("append").parquet("data/stream_output/")

## 5. Start Streaming Query

In [9]:
query = (
    stream_df
    .writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", "data/stream_checkpoint/")
    .trigger(processingTime="5 seconds")   # check for new files every 5 s
    .start()
)

print(f"Query ID : {query.id}")
print(f"Status   : {query.status['message']}")
print("\nNow run the next cell to feed test data into the stream.")

Query ID : cc19834d-6c86-4bd7-b14a-02c6b7aad44b
Status   : Initializing sources

Now run the next cell to feed test data into the stream.


## 6. Feed Test Data (simulate incoming stream)

In [ ]:
# Feed files from the HOLDOUT (unseen by classifiers) one at a time
# Run prepare_stream_holdout.ipynb first to create data/stream_holdout/

source_files = sorted(glob.glob("data/stream_holdout/part-*.parquet"))

if not source_files:
    print("Holdout not found. Run prepare_stream_holdout.ipynb first.")
else:
    print(f"Feeding {len(source_files)} holdout windows (one every 8 seconds)...\n")
    for i, src in enumerate(source_files):
        dest = f"data/stream_input/{os.path.basename(src)}"
        shutil.copy(src, dest)
        print(f"  [{i+1}/{len(source_files)}] Dropped → {os.path.basename(src)}")
        if i < len(source_files) - 1:
            time.sleep(8)
    print("\nAll holdout windows delivered.")

Feeding 20 holdout windows (one every 8 seconds)...

  [1/20] Dropped → part-00000-9010f623-fbed-4ebc-9fcc-10843b41a743-c000.snappy.parquet



[Batch 0] 1,527 records received


2026-05-03 02:30:43,109 INFO XGBoost-PySpark: predict_udf CUDF or Cupy is unavailable, fallback the inference on the CPUs
                                                                                

+----------+-----------+----------+---------------+
|Label     |label_index|prediction|predicted_class|
+----------+-----------+----------+---------------+
|BruteForce|0.0        |0.0       |BruteForce     |
|Botnet    |2.0        |2.0       |Botnet         |
|Recon     |5.0        |5.0       |Recon          |
|Botnet    |2.0        |2.0       |Botnet         |
|Normal    |4.0        |4.0       |Normal         |
|Normal    |4.0        |4.0       |Normal         |
|BruteForce|0.0        |0.0       |BruteForce     |
|Normal    |4.0        |4.0       |Normal         |
|DoS/DDoS  |3.0        |3.0       |DoS/DDoS       |
|WebAttack |1.0        |1.0       |WebAttack      |
+----------+-----------+----------+---------------+
only showing top 10 rows


2026-05-03 02:30:44,209 INFO XGBoost-PySpark: predict_udf CUDF or Cupy is unavailable, fallback the inference on the CPUs
                                                                                

...

+----------+-----------+----------+---------------+
|Label     |label_index|prediction|predicted_class|
+----------+-----------+----------+---------------+
|WebAttack |1.0        |1.0       |WebAttack      |
|Botnet    |2.0        |2.0       |Botnet         |
|BruteForce|0.0        |0.0       |BruteForce     |
|Botnet    |2.0        |2.0       |Botnet         |
|Normal    |4.0        |4.0       |Normal         |
|Recon     |5.0        |5.0       |Recon          |
|Heartbleed|7.0        |7.0       |Heartbleed     |
|WebAttack |1.0        |1.0       |WebAttack      |
|Recon     |5.0        |5.0       |Recon          |
|Botnet    |2.0        |2.0       |Botnet         |
+----------+-----------+----------+---------------+
only showing top 10 rows


2026-05-03 02:32:41,348 INFO XGBoost-PySpark: predict_udf CUDF or Cupy is unavailable, fallback the inference on the CPUs
                                                                                

  Accuracy : 0.9980  (1,522/1,525 correct)
  Latency  : 1.23s for 1,525 records  (1,238 rec/s)
  Prediction distribution:
+---------------+-----+
|predicted_class|count|
+---------------+-----+
|BruteForce     |307  |
|Recon          |241  |
|WebAttack      |232  |
|Botnet         |232  |
|DoS/DDoS       |222  |
|Normal         |210  |
|Heartbleed     |41   |
|Infiltration   |40   |
+---------------+-----+



2026-05-03 02:32:41,560 INFO XGBoost-PySpark: predict_udf CUDF or Cupy is unavailable, fallback the inference on the CPUs
2026-05-03 02:32:41,759 INFO XGBoost-PySpark: predict_udf CUDF or Cupy is unavailable, fallback the inference on the CPUs


  [17/20] Dropped → part-00016-9010f623-fbed-4ebc-9fcc-10843b41a743-c000.snappy.parquet


## 7. Monitor Query Status

In [ ]:
# Re-run this cell at any time to check progress
print("Active  :", query.isActive)
print("Status  :", query.status["message"])
print()
print("Recent batches:")
for p in query.recentProgress[-5:]:
    print(f"  batchId={p.get('batchId'):>3}  "
          f"inputRows={p.get('numInputRows', 0):>8,}  "
          f"processingRate={p.get('processedRowsPerSecond', 0):>10,.0f} rows/s")

## 8. Aggregate Results Across All Batches

In [ ]:
# Run after all files have been processed
results_df = spark.read.parquet("data/stream_output/")

total   = results_df.count()
correct = results_df.filter(F.col("label_index") == F.col("prediction")).count()

print(f"Total records classified: {total:,}")
print(f"Overall accuracy        : {correct/total:.4f}  ({correct:,}/{total:,})")
print()
print("Per-class breakdown (true vs predicted):")
results_df.groupBy("Label", "predicted_class") \
          .count() \
          .orderBy("Label") \
          .show(50, truncate=False)

## 9. Stop Stream

In [ ]:
query.stop()
print("Stream stopped.")
spark.stop()